In [81]:
import math
import random

DAYS_PER_YEAR = 252
DAYS_PER_WEEK = 5
STEPS_PER_DAY = 4

def simulate_prices(days: int):
    prices_len = STEPS_PER_DAY * days
    prices = [50]
    dt = 1 / (DAYS_PER_YEAR * STEPS_PER_DAY)
    sigma = 2.51  # 251% volatility
    for _ in range(prices_len):
        z = random.gauss(0, 1)
        # GBM step with zero drift
        prev = prices[-1]
        next_price = prev * math.exp(-0.5 * sigma**2 * dt + sigma * math.sqrt(dt) * z)
        prices.append(next_price)
    return prices

def simulate_final_prices(days: int, n_sims: int):
    finals = []
    for _ in range(n_sims):
        path = simulate_prices(days)
        finals.append(path[-1])
    return finals

def call_payoff(final_price: float, strike: float):
    return max(final_price - strike, 0.0)

def put_payoff(final_price: float, strike: float):
    return max(strike - final_price, 0.0)

def evaluate_call_price(option_price: float, finals, strike: float):
    # Estimates E[payoff - option_price]
    total = 0.0
    for S_T in finals:
        payoff = call_payoff(S_T, strike)
        total += (payoff - option_price)
    return total / len(finals)

def evaluate_put_price(option_price: float, finals, strike: float):
    # Estimates E[payoff - option_price]
    total = 0.0
    for S_T in finals:
        payoff = put_payoff(S_T, strike)
        total += (payoff - option_price)
    return total / len(finals)

def find_call_option_price(finals, strike: float, iterations: int = 20):
    low, high = 0.0, 1e3  # initial bounds
    for _ in range(iterations):
        mid = 0.5 * (low + high)
        value = evaluate_call_price(mid, finals, strike)
        if value > 0:
            low = mid
        else:
            high = mid
    return 0.5 * (low + high)

def find_put_option_price(finals, strike: float, iterations: int = 20):
    low, high = 0.0, 1e3 # initial bounds
    for _ in range(iterations):
        mid = 0.5 * (low + high)
        value = evaluate_put_price(mid, finals, strike)
        if value > 0:
            low = mid
        else:
            high = mid
    return 0.5 * (low + high)

def probability_call_itm(finals, strike: float):
    count = sum(1 for S_T in finals if S_T > strike)
    return count / len(finals)

def probability_put_itm(finals, strike: float):
    count = sum(1 for S_T in finals if S_T < strike)
    return count / len(finals)

def analyze_option(days: int, strike: int, n_sims: int, is_call: bool, print_output: bool = False):
    finals = simulate_final_prices(days, n_sims)
    if is_call:
        prob = probability_call_itm(finals, strike)
        price = find_call_option_price(finals, strike)
    else:
        prob = probability_put_itm(finals, strike)
        price = find_put_option_price(finals, strike)
    is_call_str = "Call" if is_call else "Put"
    
    if print_output:
        print(f"---")
        print(f"Option: Strike {strike} {is_call_str}, {days} Days")
        print(f"Probability ITM: {prob:.4f}")
        print(f"Estimated option price: {price:.4f}")
    return prob, price

In [82]:
# get ts going
tte_list = [3 * DAYS_PER_WEEK for _ in range(8)]
tte_list[6] -= DAYS_PER_WEEK
tte_list[7] -= DAYS_PER_WEEK
strike_list = [50, 50, 35, 40, 45, 60, 50, 50]
is_call_list = [False, True, False, False, False, True, False, True]

In [83]:
# normal option analysis, no exotic shit yet

# SIMS = 500000
# for i in range(8):
#     analyze_option(tte_list[i], strike_list[i], SIMS, is_call_list[i])

In [84]:
# only the 2 week options seem mispriced... but there's no way to get a distribution!

import numpy as np
from statistics import NormalDist

def get_distr(days: int, strike: int, m_sims: int, n_sims: int, is_call: bool):
    probs = np.array([])
    prices = np.array([])
    for _ in range(m_sims):
        prob, price = analyze_option(days, strike, n_sims, is_call)
        probs = np.append(probs, prob)
        prices = np.append(prices, price)
    return probs.mean(), probs.std(), prices.mean(), prices.std()

def price_conf_int(days: int, strike: int, m_sims: int, n_sims: int, is_call: bool, conf: float, print_output: bool = False):
    _, _, s_mean, s_std = get_distr(days, strike, m_sims, n_sims, is_call)
    z_score = NormalDist().inv_cdf(conf + (1 - conf) / 2)
    is_call_str = "Call" if is_call else "Put"
    low = s_mean - z_score * s_std
    high = s_mean + z_score * s_std
    
    if print_output:
        print(f"---")
        print(f"Option: Strike {strike} {is_call_str}, {days} Days")
        print(f"Low:  {low}")
        print(f"High: {high}")
    return low, high

In [85]:
# there's market type shit
bids = [12, 12, 4.33, 6.5, 9.05, 8.8, 9.7, 9.7]
asks = [12.05, 12.05, 4.35, 6.55, 9.1, 8.85, 9.75, 9.75]

In [88]:
M_SIMS = 100
N_SIMS = 1000

for i in range(8):
    _, _, s_mean, s_std = get_distr(tte_list[i], strike_list[i], M_SIMS, N_SIMS, is_call_list[i])
    target_bid, target_ask = bids[i], asks[i]
    low_z = (target_bid - s_mean) / s_std
    high_z = (target_ask - s_mean) / s_std
    low_prob = NormalDist().cdf(low_z)
    high_prob = 1 - NormalDist().cdf(high_z)

    print(f"---")
    print(f"Option {i}")
    print(f"{round(s_mean, 2)}, {round(s_std, 2)}")
    print(f"{round(low_prob, 2)}, {round(high_prob, 2)}")

---
Option 0
12.02, 0.37
0.48, 0.47
---
Option 1
11.99, 0.86
0.5, 0.47
---
Option 2
4.35, 0.22
0.47, 0.49
---
Option 3
6.48, 0.28
0.53, 0.4
---
Option 4
9.08, 0.33
0.46, 0.48
---
Option 5
8.82, 0.8
0.49, 0.48
---
Option 6
9.83, 0.36
0.36, 0.59
---
Option 7
9.9, 0.57
0.37, 0.6
